# Orca Nano — DPO Safety + Honesty Fix (Kaggle, on top of v4)

Continues training from the **v4 SFT checkpoint's LoRA adapter** (0.737 genesis eval,
63.0/100 golden eval — the best nano so far) using DPO instead of more SFT, to fix
two specific regressions discovered by real eval/red-team runs, not guessed at:

1. **Honesty/hedging cluster near-zero** (RBI repo rate, cash-salary legality, exact
   GST rates) — an SFT oversample attempt (boosting honesty examples to 27% of
   training data) made this WORSE (0.627) by over-generalizing hedging into
   unrelated Hindi business answers that used to score 1.0. DPO is the right tool
   here because it teaches a PREFERENCE (confident-wrong is worse than honest-hedge)
   on the specific prompts where it matters, not a broad style shift.

2. **Jailbreak resistance regressed from fine-tuning itself** — base Qwen2.5-7B
   scored 64.0/100 safety (40% jailbreak block rate); the v4 fine-tuned checkpoint
   dropped to 42.0/100 (20% block rate) and picked up a new 50% bias-flag rate that
   didn't exist in the base model. The "rejected" side of these safety pairs is
   nano-v4's OWN real captured failures on adversarial prompts (not synthetic
   harmful content asked of any teacher) — a genuine self-distillation-style
   safety correction.

**Honest scope**: ~350 total preference pairs (200 honesty + up to 150 safety,
exact safety count depends on how many real gaps existed). Small for DPO but this
is a targeted correction on top of an already-trained checkpoint, not training
from scratch — DPO needs far fewer steps than SFT to shift a specific preference.

Same hardened pipeline as the SFT notebooks: T4-forced accelerator, GGUF export
via /tmp (disk-space fix), no mid-training checkpointing (pickling bug), adapter
saved immediately after training completes.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

!pip install -q unsloth trl transformers datasets peft bitsandbytes accelerate

## Load the two preference-pair datasets and the v4 adapter

Upload three things as a Kaggle dataset before running this notebook:
- `nano_honesty_dpo.jsonl` (200 pairs)
- `nano_safety_dpo.jsonl` (safety pairs)
- `v4_adapter/` folder (adapter_config.json + adapter_model.safetensors from the v4 SFT run)

In [ ]:
import glob, json

honesty_matches = glob.glob('/kaggle/input/**/nano_honesty_dpo*.jsonl', recursive=True)
safety_matches   = glob.glob('/kaggle/input/**/nano_safety_dpo*.jsonl', recursive=True)
adapter_matches  = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)

print('Honesty pairs file:', honesty_matches)
print('Safety pairs file:', safety_matches)
print('Adapter config:', adapter_matches)

if not honesty_matches or not safety_matches or not adapter_matches:
    raise FileNotFoundError(
        "Missing one of: honesty pairs jsonl, safety pairs jsonl, or v4 adapter. "
        "Make sure all three are attached via 'Add Input'."
    )

honesty_path = honesty_matches[0]
safety_path = safety_matches[0]
adapter_dir = adapter_matches[0].rsplit('/', 1)[0]
print('Adapter dir:', adapter_dir)

In [ ]:
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records

honesty_pairs = load_jsonl(honesty_path)
safety_pairs = load_jsonl(safety_path)
all_pairs = honesty_pairs + safety_pairs

print(f'honesty pairs: {len(honesty_pairs)}')
print(f'safety pairs: {len(safety_pairs)}')
print(f'combined: {len(all_pairs)}')

## Load base model (4-bit) + attach the v4 LoRA adapter (continuing from it, not from scratch)

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

max_seq_length = 2048
base_model = "unsloth/Qwen2.5-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Attach the v4 SFT adapter, trainable — DPO continues from where SFT left off
# instead of starting from a bare base model.
model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=True)
print("[adapter] v4 LoRA adapter loaded and trainable")

## Build the preference dataset in TRL's expected format

`{"prompt", "chosen", "rejected"}` — already the exact format both pair-generation
scripts wrote, no reformatting needed.

In [ ]:
from datasets import Dataset

dpo_ds = Dataset.from_list([
    {"prompt": r["prompt"], "chosen": r["chosen"], "rejected": r["rejected"]}
    for r in all_pairs
])
print(f'dpo_ds = {len(dpo_ds)} pairs')

## DPO training

Beta 0.1 (standard), 2 epochs — DPO needs far fewer passes than SFT since it's
correcting a specific preference on top of an already-capable checkpoint, not
learning new behavior from scratch. `average_tokens_across_devices` isn't a
DPOConfig field (that bug was TrainingArguments/SFT-specific) — not needed here.
No mid-training checkpointing, same reasoning as the SFT notebooks (a real
Kaggle-side pickling bug hit that path before) — the adapter-save cell right
after training is the safety net instead.

In [ ]:
from trl import DPOConfig, DPOTrainer
import time

dpo_config = DPOConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=5e-5,
    beta=0.1,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="no",
    output_dir="/kaggle/working/output",
    report_to="none",
    max_length=1024,
    max_prompt_length=512,
)

trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
)

print("[dpo] starting DPO training...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[dpo] done in {elapsed:.1f} min")

## Save the DPO-updated adapter immediately (before merge/export)

In [ ]:
adapter_out_dir = "/kaggle/working/adapter_dpo"
model.save_pretrained(adapter_out_dir)
tokenizer.save_pretrained(adapter_out_dir)
print(f"[adapter] saved to {adapter_out_dir} — DPO-corrected weights are now safe on disk.")
!ls -la {adapter_out_dir}

## Merge LoRA + export GGUF (done in /tmp, not /kaggle/working)

Same disk-space fix as the SFT notebooks: `/kaggle/working/` has a 19.5GB quota
that a merged 16-bit 7B model + F16 GGUF intermediate would exceed.

In [ ]:
import shutil

shutil.rmtree("/tmp/merged", ignore_errors=True)
shutil.rmtree("/tmp/gguf", ignore_errors=True)

print("[merge] merging LoRA adapters (in /tmp)...")
model.save_pretrained_merged("/tmp/merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to /tmp/merged")

print("[gguf] converting to GGUF q4_k_m (in /tmp)...")
model.save_pretrained_gguf("/tmp/gguf", tokenizer, quantization_method="q4_k_m")
print("[gguf] saved under /tmp")

In [ ]:
import glob, shutil, os

candidates = [f for f in glob.glob('/tmp/**/*.gguf', recursive=True) if 'q4_k_m' in f.lower()]
print('Found in /tmp:', candidates)

if candidates:
    source_path = candidates[0]
    filename = os.path.basename(source_path)
    dest_path = f'/kaggle/working/{filename}'
    shutil.copy(source_path, dest_path)
    print(f"[export] copied to {dest_path}")
    print("\nNext: click 'Save Version' -> 'Save & Run All (Commit)' at the top right.")
else:
    print('No GGUF file found under /tmp — check the [gguf] cell above for errors.')
    print('If DPO training + adapter save both succeeded, the corrected weights are still')
    print('safe in /kaggle/working/adapter_dpo — you can retry just this export cell.')